In [2]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc


args = dict(
  model_name_or_path="meta-llama/Llama-3.2-1B-Instruct", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  adapter_name_or_path="///lf/LLaMA-Factory/legal_dpo/checkpoint-120",                        # load the saved LoRA adapters
  template="llama3",                                         # same to the one in training
  finetuning_type="lora",                                    # same to the one in training
)
chat_model = ChatModel(args)

[INFO|tokenization_utils_base.py:2095] 2025-12-15 06:51:37,702 >> loading file tokenizer.json from cache at /home/smaniyar_umass_edu/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/tokenizer.json
[INFO|tokenization_utils_base.py:2095] 2025-12-15 06:51:37,704 >> loading file tokenizer.model from cache at None
[INFO|tokenization_utils_base.py:2095] 2025-12-15 06:51:37,705 >> loading file added_tokens.json from cache at None
[INFO|tokenization_utils_base.py:2095] 2025-12-15 06:51:37,706 >> loading file special_tokens_map.json from cache at /home/smaniyar_umass_edu/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/special_tokens_map.json
[INFO|tokenization_utils_base.py:2095] 2025-12-15 06:51:37,707 >> loading file tokenizer_config.json from cache at /home/smaniyar_umass_edu/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/921317672

[INFO|2025-12-15 06:51:38] llamafactory.data.template:143 >> Add pad token: <|eot_id|>
[INFO|2025-12-15 06:51:38] llamafactory.data.template:143 >> Add <|eom_id|> to stop words.


[INFO|configuration_utils.py:765] 2025-12-15 06:51:38,885 >> loading configuration file config.json from cache at /home/smaniyar_umass_edu/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/config.json
[INFO|configuration_utils.py:839] 2025-12-15 06:51:38,887 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.

[INFO|2025-12-15 06:51:38] llamafactory.model.model_utils.kv_cache:143 >> KV cache is enabled for faster generation.


[WARNING|logging.py:328] 2025-12-15 06:51:39,722 >> `torch_dtype` is deprecated! Use `dtype` instead!
[INFO|modeling_utils.py:1172] 2025-12-15 06:51:39,725 >> loading weights file model.safetensors from cache at /home/smaniyar_umass_edu/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/model.safetensors
[INFO|modeling_utils.py:2341] 2025-12-15 06:51:39,727 >> Instantiating LlamaForCausalLM model under default dtype torch.bfloat16.
[INFO|configuration_utils.py:986] 2025-12-15 06:51:39,729 >> Generate config GenerationConfig {
  "bos_token_id": 128000,
  "eos_token_id": [
    128001,
    128008,
    128009
  ]
}

[INFO|configuration_utils.py:941] 2025-12-15 06:51:40,525 >> loading configuration file generation_config.json from cache at /home/smaniyar_umass_edu/.cache/huggingface/hub/models--meta-llama--Llama-3.2-1B-Instruct/snapshots/9213176726f574b556790deb65791e0c5aa438b6/generation_config.json
[INFO|configuration_utils.

[INFO|2025-12-15 06:51:40] llamafactory.model.model_utils.attention:143 >> Using torch SDPA for faster training and inference.
[INFO|2025-12-15 06:51:41] llamafactory.model.adapter:143 >> Merged 1 adapter(s).
[INFO|2025-12-15 06:51:41] llamafactory.model.adapter:143 >> Loaded adapter(s): ///lf/LLaMA-Factory/legal_dpo/checkpoint-120
[INFO|2025-12-15 06:51:41] llamafactory.model.loader:143 >> all params: 1,235,814,400


In [4]:
import json
import os
import logging
from tqdm import tqdm
from llamafactory.extras.misc import torch_gc
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc
import json, os, logging
from datetime import datetime
# ============================================
# PATHS
# ============================================
MODEL_PATH = "meta-llama/Llama-3.2-1B-Instruct"
DATA_FILE = "///685/Australian_data/code/subset_test_negative_best_prompt.json"  # Your positive test JSON
LOG_DIR = "///685/Australian_data/logs/llama1b/dpo_negative_best"
os.makedirs(LOG_DIR, exist_ok=True)

log_file = os.path.join(LOG_DIR, "run.log")
summary_file = os.path.join(LOG_DIR, "generated_answers.json")

# ============================================
# LOGGING SETUP
# ============================================
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
)
print(f"[INFO] Logging to: {log_file}")

# ============================================
# LOAD DATA
# ============================================
print(f"[INFO] Loading data from {DATA_FILE} ...")
with open(DATA_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)
print(f"[INFO] Loaded {len(data)} entries.\n")


# ============================================
# INFERENCE FUNCTION
# ============================================
def run_inference(question_id, entry):
    """Run model inference for a single entry."""
    prompt_text = entry.get("prompt", "")
    messages = [{"role": "user", "content": prompt_text}]
    
    try:
        response = chat_model.chat(messages)  # Replace with your model call
    except Exception as e:
        logging.error(f"[{question_id}] ERROR: {e}")
        return {"generated_answer": None, "error": str(e)}
    
    # Normalize response
    if isinstance(response, list):
        raw_text = response[0].get("content", "") if isinstance(response[0], dict) else str(response[0])
    elif isinstance(response, dict):
        raw_text = response.get("content", "")
    else:
        raw_text = str(response)
    
    # Save logs: question, prompt, and raw answer
    logging.info(f"[{question_id}] QUESTION: {entry.get('question', '')}")
    logging.info(f"[{question_id}] PROMPT:\n{prompt_text}")
    logging.info(f"[{question_id}] GENERATED ANSWER:\n{raw_text.strip()}\n")
    
    torch_gc()
    return {"generated_answer": raw_text.strip()}

# ============================================
# RUN INFERENCE ON ALL ENTRIES
# ============================================

print("\n🚀 Starting inference on test set...\n")
results = {}

for question_id in tqdm(list(data.keys()), desc="Running Inference", ncols=80):
    entry = data[question_id]
    result = run_inference(question_id, entry)
    results[question_id] = result
    
    # Flush logs after each entry
    for handler in logging.getLogger().handlers:
        handler.flush()

# ============================================
# SAVE GENERATED ANSWERS
# ============================================
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ All generated answers saved to {summary_file}")
print(f"🪵 Logs written to {log_file}")
logging.info("===  test inference run completed ===")

[INFO] Logging to: ///685/Australian_data/logs/llama1b/dpo_negative_best/run.log
[INFO] Loading data from ///685/Australian_data/code/subset_test_negative_best_prompt.json ...
[INFO] Loaded 158 entries.


🚀 Starting inference on test set...



Running Inference: 100%|██████████████████████| 158/158 [06:39<00:00,  2.53s/it]


✅ All generated answers saved to ///685/Australian_data/logs/llama1b/dpo_negative_best/generated_answers.json
🪵 Logs written to ///685/Australian_data/logs/llama1b/dpo_negative_best/run.log
